# 01 — OLLAMA Setup & Kiểm Tra Kết Nối

**Vai trò:** Model Engineer · **Task:** S1-ME-02 (Yêu cầu 9.4, 5.2, 5.4)

Notebook này minh hoạ `OllamaClient` (S1-ME-01): kiểm tra server qua `is_available()` (Yêu cầu 5.2 — không bao giờ ném exception), lấy danh sách model đã pull qua `list_models()` (Yêu cầu 5.4), và hướng dẫn pull model cần thiết — bước chuẩn bị trước khi `OllamaClient.generate()` được gọi trong `RAGPipeline.query()`.

In [1]:
import sys 
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parents[1]

if str(PROJECT_ROOT) not in sys.path:
  sys.path.insert(0, str(PROJECT_ROOT))

from src.generation.llm_client import OllamaClient
from config.settings import AppConfig

print(f"Project root: {PROJECT_ROOT}")

Project root: D:\lh222k\AI-Research-Assistant-with-RAG


## 1. Cấu hình từ AppConfig

`AppConfig.from_env()` đọc `.env` nếu có, fallback về giá trị mặc định khi biến môi trường không định nghĩa (Yêu cầu 11.3). Cell này an toàn chạy không cần file `.env`.

In [2]:
cfg = AppConfig.from_env()

print("Cấu hình OLLAMA (AppConfig):")
print(f"  base_url                : {cfg.ollama.base_url}")
print(f"  default_llm_model       : {cfg.ollama.default_llm_model}")
print(f"  default_embedding_model : {cfg.ollama.default_embedding_model}")
print(f"  timeout_seconds         : {cfg.ollama.timeout_seconds}")

Cấu hình OLLAMA (AppConfig):
  base_url                : http://localhost:11434
  default_llm_model       : llama3
  default_embedding_model : nomic-embed-text
  timeout_seconds         : 120


## 2. Hướng dẫn cài đặt OLLAMA và pull model

> Cell này là Markdown — chỉ đọc, không chạy code.

### Máy cá nhân (Windows / macOS / Linux)

```bash
# 1. Tải và cài từ https://ollama.com

# 2. Pull LLM model mặc định (vicuna:7b-v1.5-q5_1 theo AppConfig)
ollama pull vicuna:7b-v1.5-q5_1

# 3. Pull embedding model mặc định (bge-m3:latest theo AppConfig)
ollama pull bge-m3:latest

# 4. Kiểm tra
ollama list
```

### Google Colab

```python
import os, subprocess, time
!sudo apt-get install -y zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh
os.environ['OLLAMA_FLASH_ATTENTION'] = 'false'
subprocess.Popen(["ollama", "serve"]); time.sleep(30)
!ollama pull vicuna:7b-v1.5-q5_1
!ollama pull bge-m3:latest
```

## 3. Khởi tạo OllamaClient và kiểm tra `is_available()` (Yêu cầu 5.2)

`is_available()` trả về `bool` và **không bao giờ ném exception** trong bất kỳ trường hợp nào — dù server chưa chạy, URL sai, hay timeout. Cell này an toàn chạy trước khi `ollama serve` được khởi động.

In [3]:
client = OllamaClient(
    model_name=cfg.ollama.default_llm_model,
    base_url=cfg.ollama.base_url,
)

# Kiểm tra is_available() trả về bool (Yêu cầu 5.2)
available = client.is_available()

print(f"OllamaClient:")
print(f"  model_name : {client.model_name}")
print(f"  base_url   : {client.base_url}")
print()
print(f"is_available() => {available}")

if available:
    print("✅ Server đang chạy — tiếp tục cell bên dưới.")
else:
    print("⚠️  Server chưa chạy — chạy 'ollama serve' trên terminal.")
    print("   Notebook vẫn tiếp tục không lỗi (Yêu cầu 9.4).")

OllamaClient:
  model_name : llama3
  base_url   : http://localhost:11434

is_available() => True
✅ Server đang chạy — tiếp tục cell bên dưới.


## 4. Lấy danh sách model với `list_models()` (Yêu cầu 5.4)

`list_models()` trả về `List[str]` tên các model đã pull, hoặc `[]` nếu server không chạy hay response không hợp lệ — không bao giờ ném exception, đảm bảo notebook chạy hết cell an toàn (Yêu cầu 9.4).

In [4]:
pulled_models = client.list_models()

if pulled_models:
    print(f"Danh sách model đã pull ({len(pulled_models)} model):")
    for i, name in enumerate(pulled_models, 1):
        tag = " -> LLM mặc định" if name.startswith(cfg.ollama.default_llm_model.split(":")[0]) else ""
        tag = " -> Embed mặc định" if name.startswith(cfg.ollama.default_embedding_model.split(":")[0]) else tag
        print(f"  {i:2d}. {name}{tag}")
else:
    print("Không có model hoặc server chưa chạy — xem hướng dẫn ở mục 2.")

Danh sách model đã pull (2 model):
   1. llama3:latest -> LLM mặc định
   2. nomic-embed-text:latest -> Embed mặc định


## 5. Kiểm tra model mặc định đã pull chưa

So sánh danh sách từ `list_models()` với `default_llm_model` và `default_embedding_model` trong `AppConfig` để biết pipeline có thể chạy đầy đủ hay chưa.

In [5]:
if not available:
    print("⚠️  Bỏ qua — server chưa chạy.")
else:
    llm_prefix   = cfg.ollama.default_llm_model.split(":")[0]
    embed_prefix = cfg.ollama.default_embedding_model.split(":")[0]

    llm_ready   = any(m.startswith(llm_prefix)   for m in pulled_models)
    embed_ready = any(m.startswith(embed_prefix) for m in pulled_models)

    print("Kiểm tra model cần thiết cho RAGPipeline:")
    print(f"  LLM   '{cfg.ollama.default_llm_model}'   : {'✅ Đã pull' if llm_ready else '❌ Chưa pull'}")
    print(f"  Embed '{cfg.ollama.default_embedding_model}' : {'✅ Đã pull' if embed_ready else '❌ Chưa pull'}")

    if not llm_ready:
        print(f"\n  → ollama pull {cfg.ollama.default_llm_model}")
    if not embed_ready:
        print(f"  → ollama pull {cfg.ollama.default_embedding_model}")
    if llm_ready and embed_ready:
        print("\n  ✅ Đủ model — RAGPipeline sẵn sàng chạy.")

Kiểm tra model cần thiết cho RAGPipeline:
  LLM   'llama3'   : ✅ Đã pull
  Embed 'nomic-embed-text' : ✅ Đã pull

  ✅ Đủ model — RAGPipeline sẵn sàng chạy.


## 6. Tổng kết

- `OllamaClient.is_available()` trả về `bool` và không bao giờ ném exception dù server tắt — đảm bảo `RAGPipeline` có thể kiểm tra trạng thái an toàn trước khi gọi `generate()` (Yêu cầu 5.2).
- `OllamaClient.list_models()` trả về `List[str]` tên model đã pull, hoặc `[]` khi server không chạy — không ném exception, notebook chạy an toàn dù OLLAMA offline (Yêu cầu 5.4, 9.4).
- `OllamaClient.generate()` và `generate_stream()` raise `NotImplementedError` để fail-fast và đúng contract `BaseLLMClient` — sẽ triển khai ở Sprint 3.
- Notebook tiếp theo: `02_model_comparison.ipynb` — so sánh hiệu năng các LLM model.